In [0]:
spark.sql("CREATE CATALOG IF NOT EXISTS fraud_detection COMMENT 'Fraud detection pipeline'")
spark.sql("USE CATALOG fraud_detection")
display(spark.sql("SHOW CATALOGS"))

In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS fraud_detection.bronze COMMENT 'Raw ingested data'")
spark.sql("CREATE SCHEMA IF NOT EXISTS fraud_detection.silver COMMENT 'Cleaned, Joined data'")
spark.sql("CREATE SCHEMA IF NOT EXISTS fraud_detection.gold COMMENT 'Aggregated business-ready data'")

display(spark.sql("SHOW SCHEMAS"))

In [0]:
spark.sql("""
          CREATE VOLUME IF NOT EXISTS fraud_detection.bronze.raw_files
          COMMENT 'Landing zone for CSV drops and static reference files'
          """)

display(dbutils.fs.ls("/Volumes/fraud_detection/bronze/raw_files/"))

In [0]:
display(spark.sql("SHOW VOLUMES IN fraud_detection.bronze"))

In [0]:
df = spark.read.csv("/Volumes/fraud_detection/bronze/raw_files/*.csv", 
                    header=True, 
                    inferSchema=True,
                    multiLine=True,
                    escape='"')
display(df.limit(10))
print(f"Row count: {df.count()}")

In [0]:
from pyspark.sql.functions import col
# change column names to snake_case and write the data to bronze
df_renamed = df.select(
    col("Transaction ID").alias("transaction_id"),
    col("Customer ID").alias("customer_id"),
    col("Transaction Amount").alias("transaction_amount"),
    col("Transaction Date").alias("transaction_date"),
    col("Payment Method").alias("payment_method"),
    col("Product Category").alias("product_category"),
    col("Quantity").alias("quantity"),
    col("Customer Age").alias("customer_age"),
    col("Customer Location").alias("customer_location"),
    col("Device Used").alias("device_used"),
    col("IP Address").alias("ip_address"),
    col("Shipping Address").alias("shipping_address"),
    col("Billing Address").alias("billing_address"),
    col("Is Fraudulent").alias("is_fraudulent"),
    col("Account Age Days").alias("account_age_days"),
    col("Transaction Hour").alias("transaction_hour")
)

df_renamed.write.mode("overwrite").saveAsTable("fraud_detection.bronze.transactions_raw")

display(spark.sql("SELECT COUNT(*) AS row_count FROM fraud_detection.bronze.transactions_raw"))